# AI工学101 — 第6回
## 統計量とデータ前処理：モデルに食わせる前にデータを整える

前回は

\[
y = XW+b
\]

まで到達しました。今日はその `X` のほうを料理します。🍳

機械学習ではモデルそのものと同じくらい、**入力データをどう整えるか**が重要です。

**所要時間：60〜90分**  
講義：約20分 ／ 実習：約40〜50分 ／ 演習：約20分

---

## 📖 講義：なぜ前処理が必要なのか

こんな身体データを考えてみます。

```python
X = np.array([
    [20, 160, 50],
    [30, 170, 65],
    [40, 180, 80]
])
```

各列は、

```text
年齢 | 身長(cm) | 体重(kg)
```

です。

ところが値のスケールが違います。

```text
年齢     20〜40
身長    160〜180
体重     50〜80
```

もっと極端に、

```text
年齢           30
年収     5,000,000
```

だったらどうでしょう。

数値の大きさがまるで違います。

機械学習アルゴリズムによっては、この違いによって学習が不安定になったり、特定の特徴量が過剰に影響したりします。

そこで登場するのが**標準化**です。

---

# 💻 実習1：平均

まずNumPyで平均を計算します。

```python
import numpy as np

x = np.array([10, 20, 30, 40, 50])

print(x.mean())
```

結果：

```text
30.0
```

これは、

\[
\bar{x}=\frac{1}{n}\sum_{i=1}^{n}x_i
\]

ですね。

NumPyなら一行。

```python
np.mean(x)
```

でも同じです。

---

# 💻 実習2：平均との差

平均からどれだけ離れているか計算します。

```python
mean = x.mean()

centered = x - mean

print(centered)
```

結果：

```text
[-20. -10.   0.  10.  20.]
```

ここでは第2回の**ブロードキャスト**が働いています。

そして確認。

```python
print(centered.mean())
```

ほぼ、

```text
0
```

になります。

これを**中心化（centering）**といいます。

---

# 📖 分散とは？

平均との差だけでは、データ全体がどれくらい散らばっているか分かりません。

そこで、

\[
\mathrm{Var}(X)
=
\frac{1}{N}
\sum_{i=1}^{N}(x_i-\bar{x})^2
\]

を計算します。

つまり、

```text
平均との差
 ↓
二乗
 ↓
平均
```

です。

---

# 💻 実習3：分散を自作する

NumPyの `var()` を使う前に、自分で書きます。

```python
diff = x - x.mean()

squared = diff ** 2

variance = squared.mean()

print(variance)
```

そして、

```python
print(x.var())
```

を実行。

同じ値になるはずです。

ここ大事。

`np.var()` は魔法ではなく、

```python
((x - x.mean()) ** 2).mean()
```

をやっているだけです。

---

# 💻 実習4：標準偏差

分散にはちょっと困ったところがあります。

元データが、

```text
cm
```

なら、分散は、

```text
cm²
```

になります。

そこで平方根を取ります。

\[
\sigma=\sqrt{\mathrm{Var}(X)}
\]

NumPyでは、

```python
std = x.std()

print(std)
```

これが**標準偏差（standard deviation）**です。

---

# 📖 今日の主役：標準化

いよいよ機械学習の前処理です。

標準化は、

\[
z=\frac{x-\mu}{\sigma}
\]

です。

難しそうに見えるけれど、

```text
平均との差
─────────
標準偏差
```

だけ。

結果として、

```text
平均 ≈ 0
標準偏差 ≈ 1
```

のデータになります。

---

# 💻 実習5：標準化を自作する

```python
x = np.array([10, 20, 30, 40, 50])

mean = x.mean()
std = x.std()

z = (x - mean) / std

print(z)
```

確認します。

```python
print(z.mean())
print(z.std())
```

だいたい、

```text
0.0
1.0
```

になれば成功。

🎉 **標準化をNumPyだけで実装できました。**

---

# 💻 実習6：機械学習っぽいデータでやる

ここから本番。

```python
X = np.array([
    [20, 160, 50],
    [30, 170, 65],
    [40, 180, 80],
    [50, 190, 95]
], dtype=float)
```

shapeは、

```text
(4, 3)
```

つまり、

```text
4 samples × 3 features
```

です。

特徴量ごとの平均を計算。

```python
mean = X.mean(axis=0)

print(mean)
```

`axis=0`。

第4回が帰ってきました。

---

続いて特徴量ごとの標準偏差。

```python
std = X.std(axis=0)

print(std)
```

そして、

```python
X_scaled = (X - mean) / std

print(X_scaled)
```

ここでは、

- `axis`
- ブロードキャスト
- 平均
- 標準偏差

が全部つながっています。

---

## 本当に標準化できた？

確認します。

```python
print(X_scaled.mean(axis=0))
```

ほぼ、

```text
[0. 0. 0.]
```

になるはず。

さらに、

```python
print(X_scaled.std(axis=0))
```

ほぼ、

```text
[1. 1. 1.]
```

になります。

よし。

**機械学習用の前処理を自力実装できました。**

---

# 💻 実習7：外れ値を見る

次は、

```python
scores = np.array([
    48, 51, 49, 52, 50, 47, 53, 200
])
```

明らかに、

```text
200
```

だけ様子がおかしい。

平均を見ます。

```python
print(scores.mean())
```

中央値も見ます。

```python
print(np.median(scores))
```

ここで、

> **平均は外れ値の影響を受けやすい**

という性質が見えます。

だから実データでは、

```text
平均だけ見る
```

では危ないことがあります。

これが後のEDA（探索的データ分析）につながります。

以前「EDAって何？」となっていたやつ、実はもう入口に立っています。

---

# ✍️ 演習：自分でデータを標準化する

次のデータを使います。

```python
X = np.array([
    [25, 150],
    [30, 180],
    [35, 170],
    [40, 200],
    [45, 160],
    [50, 220]
], dtype=float)
```

今回は、

```text
年齢 | スコア
```

とします。

課題は5つです。

1. `X.shape` を確認する。
2. 特徴量ごとの平均を `axis=0` で計算する。
3. 特徴量ごとの標準偏差を計算する。
4. NumPyだけで `X` を標準化する。
5. 標準化後について、特徴量ごとの平均と標準偏差を確認する。

最終的に、

```python
X_scaled.mean(axis=0)
```

がほぼ、

```text
[0. 0.]
```

そして、

```python
X_scaled.std(axis=0)
```

がほぼ、

```text
[1. 1.]
```

ならクリアです。👾

---

## 🌿 今日の重要ポイント

今日は関数を覚えるというより、この関係を理解してほしい。

\[
\text{平均との差}
\rightarrow
\text{分散}
\rightarrow
\text{標準偏差}
\rightarrow
\text{標準化}
\]

そして実装すると、

```python
mean = X.mean(axis=0)
std = X.std(axis=0)

X_scaled = (X - mean) / std
```

**たった3行です。**

これが後で  に進むと、

```python
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
```

になります。

でも僕らは先にNumPyで中身を実装した。

だから後で `StandardScaler` を見ても、**「便利な呪文」ではなく、中で何をやっているか説明できる**わけです。

これがAI工学101でNumPyから始めた理由でもあります。

次の**第7回は「欠損値・外れ値・データクリーニング」**。

`NaN`、`np.isnan()`、欠損値の除外・補完などを扱って、**汚い現実データをモデルに投入できる形へ変える**ところまで進めます。👨‍🏫🌿